In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    TimeSeriesSplit,
    RandomizedSearchCV
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import joblib
import warnings

warnings.filterwarnings("ignore")

In [2]:
X_train = joblib.load("Demand_X_train.pkl")
X_test = joblib.load("Demand_X_test.pkl")

y_train = joblib.load("Demand_y_train.pkl")
y_test = joblib.load("Demand_y_test.pkl")

In [3]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (100, 11)
X_test  : (26, 11)
y_train : (100,)
y_test  : (26,)


In [4]:
# Time Series Cross Validation
tscv = TimeSeriesSplit(n_splits=5)

# Random Forest Model
rf = RandomForestRegressor(random_state=42)

# Hyperparameter Search Space
param_dist = {
    'n_estimators': [100, 200, 300, 500, 700],
    'max_depth': [5, 8, 10, 15, 20, None],
    'min_samples_split': [2, 3, 5, 8],
    'min_samples_leaf': [1, 2, 3, 4],
    'max_features': ['sqrt', 'log2', 0.7, 1.0],
    'bootstrap': [True, False]
}

# Random Search
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=tscv,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

# Train Model
random_search.fit(X_train, y_train)

# Best Model
rf_model = random_search.best_estimator_

print("Best Parameters:")
print(random_search.best_params_)

Best Parameters:
{'n_estimators': 700, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 1.0, 'max_depth': 20, 'bootstrap': False}


In [5]:
rf_model.fit(X_train, y_train)

RandomForestRegressor(bootstrap=False, max_depth=20, n_estimators=700,
                      random_state=42)

In [6]:
y_pred = rf_model.predict(X_test)

In [7]:
print(y_pred)

[ 8718.20714286  9339.95142857  9606.55571429 10373.14571429
 10373.36142857 11390.00857143 11392.61857143 10920.56285714
 10858.77857143 10920.64       11393.53       10172.
  9423.09114286  9426.76514286  9766.42857143 10147.4
 11389.92571429 11396.18142857 11399.28857143 11399.28857143
 11400.03428571 11399.74428571 10922.56       10167.67142857
 10024.81857143 10154.68571429]


In [8]:
prediction_df = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})
prediction_df.head(10)

,Actual,Predicted
0,8759.0,8718.207143
1,9356.0,9339.951429
2,9658.0,9606.555714
3,10404.0,10373.145714
4,10452.0,10373.361429
5,12649.0,11390.008571
6,11838.0,11392.618571
7,10784.0,10920.562857
8,11192.0,10858.778571
9,11135.0,10920.640000


In [9]:
# Predictions
y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

# Training Metrics
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_mape = mean_absolute_percentage_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Testing Metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_mape = mean_absolute_percentage_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*50)
print("Training Performance")
print("="*50)
print(f"MAE  : {train_mae:.2f}")
print(f"RMSE : {train_rmse:.2f}")
print(f"MAPE : {train_mape*100:.2f}%")
print(f"R²   : {train_r2:.4f}")

print("\n")

print("="*50)
print("Testing Performance")
print("="*50)
print(f"MAE  : {test_mae:.2f}")
print(f"RMSE : {test_rmse:.2f}")
print(f"MAPE : {test_mape*100:.2f}%")
print(f"R²   : {test_r2:.4f}")

Training Performance
MAE  : 0.00
RMSE : 0.00
MAPE : 0.00%
R²   : 1.0000


Testing Performance
MAE  : 210.70
RMSE : 366.60
MAPE : 1.80%
R²   : 0.8610


In [10]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

               Feature  Importance
2   Electricity_Supply    0.998241
8         Demand_Lag_1    0.000268
6            Month_sin    0.000241
4          Temperature    0.000224
9         Demand_Lag_2    0.000221
10        Demand_Lag_3    0.000189
5                 Year    0.000171
0             Humidity    0.000158
1             Rainfall    0.000123
7            Month_cos    0.000108
3     Solar_Irradiance    0.000054


In [11]:
print(X_train.describe())

         Humidity    Rainfall  Electricity_Supply  Solar_Irradiance  \
count  100.000000  100.000000            100.0000        100.000000   
mean    74.160400   95.606726           9138.1740        152.503900   
std      7.578832   75.634605            944.1379         23.807573   
min     56.410000    0.786286           6508.0000         79.840000   
25%     69.722500   37.067500           8480.0000        139.827500   
50%     74.520000   82.542714           9056.5500        155.470000   
75%     79.937500  139.723929           9685.1750        167.277500   
max     89.420000  399.657143          11349.0000        204.470000   

       Temperature        Year     Month_sin     Month_cos  Demand_Lag_1  \
count   100.000000   100.00000  1.000000e+02  1.000000e+02    100.000000   
mean     26.950600  2019.16000 -3.232051e-02 -8.660254e-03   9123.030000   
std       1.801522     2.44007  7.152420e-01  7.052652e-01    932.279719   
min      23.660000  2015.00000 -1.000000e+00 -1.000000e+

In [12]:
print(X_test.describe())

        Humidity    Rainfall  Electricity_Supply  Solar_Irradiance  \
count  26.000000   26.000000           26.000000         26.000000   
mean   76.338077  107.520901        10683.769231        184.847123   
std     7.761415   76.711440         1015.390577         60.849792   
min    56.850000    4.588857         8762.000000         98.660000   
25%    72.005000   41.301643        10078.750000        128.440000   
50%    76.485000   91.552000        10601.500000        186.560000   
75%    82.562500  163.227786        11316.750000        228.634221   
max    86.440000  257.944286        12649.000000        303.305385   

       Temperature         Year     Month_sin     Month_cos  Demand_Lag_1  \
count    26.000000    26.000000  2.600000e+01  2.600000e+01     26.000000   
mean     26.957308  2024.384615 -1.923077e-02  7.177021e-02  10722.884615   
std       1.853549     0.637302  6.997252e-01  7.379993e-01    996.390057   
min      23.560000  2023.000000 -1.000000e+00 -1.000000e+00  

In [15]:
print(X_train.columns.tolist())

['Humidity', 'Rainfall', 'Electricity_Supply', 'Solar_Irradiance', 'Temperature', 'Year', 'Month_sin', 'Month_cos', 'Demand_Lag_1', 'Demand_Lag_2', 'Demand_Lag_3']


In [16]:
import joblib

# Save the trained Random Forest model
joblib.dump(rf_model, "RandomForest_Demand_Forecasting_08610.pkl")

print("Random Forest model saved successfully!")

Random Forest model saved successfully!
